# RoadWatch BD: fine-tune the detector on BadODD (Kaggle GPU)

Fine-tunes **RT-DETR r18** (Apache-2.0, COCO-pretrained) on **BadODD** (CC BY 4.0, 9,825 Bangladeshi road
images) in RoadWatch's 15 unified classes. Output: a model folder that the pipeline loads with
`--detector hf --weights <folder>`.

**Before running:** in the right-hand panel set **Accelerator → GPU T4 x2** (or P100) and turn
**Internet on** (needs a phone-verified Kaggle account). Then *Run all*.
Time: ~10 min setup (4.4 GB download), then roughly 2–4 min per epoch on a T4.

Credit when publishing results: BadODD, Baig et al., arXiv:2401.10659, Zenodo record 13823687, CC BY 4.0.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone --depth 1 https://github.com/muttakin210/roadwatch-bd.git /kaggle/working/roadwatch-bd
!pip install -q "transformers>=4.50" timm "torchmetrics>=1.4" "pycocotools>=2.0.8" opencv-python-headless pyyaml

## 1. Download BadODD from Zenodo and check its checksum

In [ ]:
%%bash
set -e
mkdir -p /kaggle/temp/badodd && cd /kaggle/temp/badodd
URL="https://zenodo.org/api/records/13823687/files/badodd.zip/content"
[ -f badodd.zip ] || wget -q -O badodd.zip "$URL"
echo "6a0d3983b379302ab79a99baf7d47d74  badodd.zip" | md5sum -c -
unzip -q -o badodd.zip -d extracted && rm badodd.zip
ls extracted

## 2. Convert to RoadWatch classes
BadODD's *Auto Rickshaw* (CNG, easybike and leguna mixed) becomes `cng_autorickshaw`; *Three-Wheeler* is the
pedal rickshaw; Train and Wheelchair are dropped. See `ml/scripts/convert_datasets.py`.

In [ ]:
%cd /kaggle/working/roadwatch-bd/ml
!python scripts/convert_datasets.py --dataset badodd --src /kaggle/temp/badodd/extracted --dst /kaggle/temp/badodd_unified

## 3. Train
The best epoch (by val mAP@0.5) is saved to `rtdetr_bd/best`. For a quick trial first, add
`--epochs 2 --max-train 500`.

In [ ]:
!python scripts/train_detector.py --data /kaggle/temp/badodd_unified --out /kaggle/working/rtdetr_bd     --epochs 30 --batch 8 --workers 4 --max-val 500

## 4. Results and download

In [ ]:
import json
log = json.load(open("/kaggle/working/rtdetr_bd/train_log.json"))
b = log.get("best", {})
print("best epoch", b.get("epoch"), "mAP50", b.get("map50"), "mAP", b.get("map"))
for c, ap in sorted(b.get("ap_per_class", {}).items(), key=lambda x: -x[1]):
    print(f"  {c:18s} AP {ap:.3f}")
!cd /kaggle/working && zip -qr rtdetr_bd_best.zip rtdetr_bd/best rtdetr_bd/train_log.json && ls -lh rtdetr_bd_best.zip

Download `rtdetr_bd_best.zip` from the **Output** panel. On your PC, unzip it into
`D:\Traffic AI model project\ml\models\` and run:

```
cd ml
python -m roadwatch.pipeline <video.mp4> --out ../data/processed/test --detector hf --weights models/rtdetr_bd/best --fresh
```

Remember: BadODD is dashcam footage. Check the model on **footbridge** clips before trusting it, and
legunas and easybikes still need your own labelled frames.